<img src="http://imgur.com/1ZcRyrc.png" style="float: left; margin: 20px; height: 55px">

# 2. Advanced Chunking, Embeddings & Indexing (Solved Reference)

Fully worked solutions to all three exercises. Use this to check your work or to
catch up if you fell behind during the live lab. See `solutions folder` for
prose explanations alongside this code.

---

## Solution Guide

1. [Exercise 1 — Heading-Path Tracking for Adaptive Chunking](#exercise1)
2. [Exercise 2 — Embedding Representation for Product Reviews ](#exercise2)
3. [Exercise 3 — Tiered Index Advisor and IVF-PQ Benchmark](#exercise3)

---

In [12]:
%pip install faiss-cpu numpy langchain-text-splitters langchain-openai langchain-huggingface langchain-pinecone pinecone

In [2]:
import os
import getpass

# Set your OpenAI API Key here if you have one. This is optional.
openai_key = getpass.getpass("Enter your OpenAI API Key (leave empty to skip): ")
if openai_key:
    os.environ["OPENAI_API_KEY"] = openai_key

# Set your Pinecone API Key here if you have one. This is optional.
pinecone_key = getpass.getpass("Enter your Pinecone API Key (leave empty to skip): ")
if pinecone_key:
    os.environ["PINECONE_API_KEY"] = pinecone_key

print("API key input prompts added.")

Enter your OpenAI API Key (leave empty to skip): ··········
Enter your Pinecone API Key (leave empty to skip): ··········
API key input prompts added.


<h2 id="exercise1"> Exercise 1 — Heading-Path Tracking for Adaptive Chunking </h2>

In [3]:
import re

CODE_FENCE_PATTERN = re.compile(r"```.*?```", re.DOTALL)
TABLE_ROW_PATTERN = re.compile(r"^\s*\|.*\|\s*$", re.MULTILINE)
HEADER_PATTERN = re.compile(r"^(#{1,6})\s+(.*)$", re.MULTILINE)

def fixed_size_chunk(text: str, chunk_size: int = 200) -> list:
    words = text.split()
    chunks, current = [], []
    current_len = 0
    for word in words:
        current.append(word)
        current_len += len(word) + 1
        if current_len >= chunk_size:
            chunks.append(" ".join(current))
            current, current_len = [], 0
    if current:
        chunks.append(" ".join(current))
    return chunks


def merge_table_spans(document: str) -> list:
    table_rows = list(TABLE_ROW_PATTERN.finditer(document))
    spans = []
    if table_rows:
        block_start = table_rows[0].start()
        block_end = table_rows[0].end()
        for row in table_rows[1:]:
            if row.start() <= block_end + 2:
                block_end = row.end()
            else:
                spans.append((block_start, block_end, "table"))
                block_start, block_end = row.start(), row.end()
        spans.append((block_start, block_end, "table"))
    return spans


def adaptive_chunk_v2(document: str) -> list:
    events = []
    for m in HEADER_PATTERN.finditer(document):
        events.append((m.start(), m.end(), "header", len(m.group(1)), m.group(2).strip()))
    for m in CODE_FENCE_PATTERN.finditer(document):
        events.append((m.start(), m.end(), "code", None, None))
    for start, end, _ in merge_table_spans(document):
        events.append((start, end, "table", None, None))

    events.sort(key=lambda e: e[0])

    chunks = []
    stack = []   # (level, title), most-specific last
    cursor = 0

    def heading_path():
        return " > ".join(title for _, title in stack) if stack else None

    def emit_prose(text_slice):
        text_slice = text_slice.strip()
        if text_slice:
            for c in fixed_size_chunk(text_slice):
                chunks.append({"text": c, "content_type": "prose", "heading_path": heading_path()})

    for start, end, kind, level, title in events:
        if start > cursor:
            emit_prose(document[cursor:start])

        if kind == "header":
            stack = [item for item in stack if item[0] < level]
            stack.append((level, title))
        else:
            chunks.append({"text": document[start:end].strip(), "content_type": kind, "heading_path": heading_path()})

        cursor = end

    if cursor < len(document):
        emit_prose(document[cursor:])

    return chunks


In [3]:
doc = """# Setup

Some intro text about setup.

## Authentication

Explains how authentication works in general.

### Token Refresh

Here is how you refresh a token.

```python
def refresh(token):
    return new_token(token)
```

## Deployment

Deployment notes go here."""

for c in adaptive_chunk_v2(doc):
    print(f"{c['content_type']:6} | {str(c['heading_path']):45} | {c['text'][:40].strip()!r}")


prose  | Setup                                         | 'Some intro text about setup.'
prose  | Setup > Authentication                        | 'Explains how authentication works in gen'
prose  | Setup > Authentication > Token Refresh        | 'Here is how you refresh a token.'
code   | Setup > Authentication > Token Refresh        | '```python\ndef refresh(token):\n    return'
prose  | Setup > Deployment                            | 'Deployment notes go here.'


<h2 id="exercise2"> Exercise 2 — Embedding Representation for Product Reviews </h2>

In [4]:
import hashlib
import numpy as np

def fake_embed(text: str, dim: int = 16) -> np.ndarray:
    vec = np.zeros(dim)
    for word in text.lower().split():
        h = int(hashlib.md5(word.encode()).hexdigest(), 16)
        vec[h % dim] += 1.0
    norm = np.linalg.norm(vec)
    return vec / norm if norm > 0 else vec


def prepare_for_embedding_v2(content: dict) -> str:
    content_type = content["type"]

    if content_type == "product_review":
        # Prepend the rating as an explicit natural-language qualifier rather than
        # appending it or dropping it: this places the sentiment-bearing word
        # ("poor"/"excellent") immediately next to the topic words the review
        # discusses, so a proximity-sensitive similarity function weighs rating
        # and content together rather than the rating getting diluted at the far
        # end of a long review body.
        rating = content["rating"]
        quality_word = {1: "poor", 2: "below average", 3: "average", 4: "good", 5: "excellent"}[rating]
        verified = "Verified purchase. " if content.get("verified") else ""
        return f"{verified}{quality_word} ({rating}/5 stars) review: {content['text']}"

    if content_type == "media_tags":
        tags = content["tags"]
        return f"A {' '.join(tags[:-1])} film featuring {tags[-1]}."

    if content_type == "paper_table":
        row = content["row"]
        return (f"On the {content['benchmark']} benchmark, "
                f"{row['model']} achieves an F1 score of {row['f1']}.")

    if content_type == "tweet":
        return content["text"]

    if content_type == "book_chapter":
        return content["chunk_text"]

    raise ValueError(f"Unknown content type: {content_type}")


In [5]:
samples = [
    {"type": "product_review", "rating": 1, "verified": True,
     "text": "Battery life is abysmal, dead by lunchtime every single day."},
    {"type": "product_review", "rating": 5, "verified": True,
     "text": "Battery easily lasts two full days, very impressed."},
]

for sample in samples:
    representation = prepare_for_embedding_v2(sample)
    print(f"representation={representation!r}")

v1 = fake_embed(prepare_for_embedding_v2(samples[0]))
v2 = fake_embed(prepare_for_embedding_v2(samples[1]))
cos_sim = float(np.dot(v1, v2))
print(f"\nCosine similarity between the 1-star and 5-star battery reviews: {cos_sim:.3f}")
assert cos_sim < 0.99, "Reviews should not be nearly identical vectors"
print("Confirmed: opposite-sentiment reviews are distinguishable in embedding space.")


representation='Verified purchase. poor (1/5 stars) review: Battery life is abysmal, dead by lunchtime every single day.'
representation='Verified purchase. excellent (5/5 stars) review: Battery easily lasts two full days, very impressed.'

Cosine similarity between the 1-star and 5-star battery reviews: 0.544
Confirmed: opposite-sentiment reviews are distinguishable in embedding space.


<h2 id="exercise3"> Exercise 3 — Tiered Index Advisor and IVF-PQ Benchmark </h2>

In [5]:
import faiss
import numpy as np

def recommend_index_v2(num_searches: int, need_exact: bool, memory_concern: str, num_vectors: int) -> str:
    if num_searches < 10_000:
        return "Flat"
    if need_exact:
        return "Flat"
    if memory_concern == "none":
        return "HNSW32"
    if memory_concern in ("some", "a_lot"):
        pq_suffix = ",PQ32" if memory_concern == "a_lot" else ",Flat"
        if num_vectors < 1_000_000:
            nlist = int(4 * num_vectors ** 0.5)
            return f"IVF{nlist}{pq_suffix}"
        elif num_vectors < 10_000_000:
            return f"IVF65536_HNSW32{pq_suffix}"
        elif num_vectors < 100_000_000:
            return f"IVF262144_HNSW32{pq_suffix}"
        else:
            return f"IVF1048576_HNSW32{pq_suffix}"
    return "OPQ32_128,IVF4096,PQ32"


for n in [50_000, 5_000_000, 50_000_000]:
    print(f"{n:>12,} vectors -> some: {recommend_index_v2(100_000, False, 'some', n):<22} "
          f"a_lot: {recommend_index_v2(100_000, False, 'a_lot', n)}")


      50,000 vectors -> some: IVF894,Flat            a_lot: IVF894,PQ32
   5,000,000 vectors -> some: IVF65536_HNSW32,Flat   a_lot: IVF65536_HNSW32,PQ32
  50,000,000 vectors -> some: IVF262144_HNSW32,Flat  a_lot: IVF262144_HNSW32,PQ32


In [7]:
def benchmark_ivf_pq(vectors: np.ndarray, dim: int, flat_ids: np.ndarray, query: np.ndarray) -> dict:
    nlist = 100            # overriding the sqrt-based formula for this small demo dataset
    m_subquantizers = 32   # divides dim=128 evenly; less aggressive than 16 -> meaningfully better recall
    quantizer = faiss.IndexFlatL2(dim)
    ivf_pq = faiss.IndexIVFPQ(quantizer, dim, nlist, m_subquantizers, 8)
    ivf_pq.train(vectors)
    ivf_pq.add(vectors)
    ivf_pq.nprobe = 20

    _, ivf_pq_ids = ivf_pq.search(query, k=10)
    recall_at_10 = len(set(flat_ids[0]) & set(ivf_pq_ids[0])) / 10.0

    bytes_per_vector_pq = ivf_pq.code_size
    bytes_per_vector_flat = dim * 4
    compression_ratio = bytes_per_vector_flat / bytes_per_vector_pq

    return {
        "bytes_per_vector": bytes_per_vector_pq,
        "compression_ratio": compression_ratio,
        "recall_at_10": recall_at_10,
    }


np.random.seed(42)
dim = 128
num_vectors = 20_000
num_clusters = 50
centers = np.random.random((num_clusters, dim)).astype("float32") * 10
labels = np.random.randint(0, num_clusters, size=num_vectors)
vectors = centers[labels] + np.random.normal(scale=0.5, size=(num_vectors, dim)).astype("float32")
query = centers[0].reshape(1, -1) + np.random.normal(scale=0.5, size=(1, dim)).astype("float32")

flat_index = faiss.IndexFlatL2(dim)
flat_index.add(vectors)
_, flat_ids = flat_index.search(query, k=10)

result = benchmark_ivf_pq(vectors, dim, flat_ids, query)
print(f"IVF-PQ bytes/vector: {result['bytes_per_vector']}")
print(f"Flat bytes/vector:   {dim * 4}")
print(f"Compression ratio:   {result['compression_ratio']:.1f}x smaller")
print(f"Recall@10 vs exact:  {result['recall_at_10']:.0%}")


IVF-PQ bytes/vector: 32
Flat bytes/vector:   512
Compression ratio:   16.0x smaller
Recall@10 vs exact:  70%


# Extensions: Local Models vs. Cloud/API Models

> **The code cells below require either a paid API key (Cloud/API track) or a local model download**

## Extension A — Local Models Approach

**Advanced Chunking, Embeddings & Indexing**

The course's labs use hash-based deterministic embeddings so every result is exactly
reproducible offline. This section shows the real *local* (self-hosted, no paid API)
tooling equivalents.

### What Changes, Component by Component

| Course concept | Course's stand-in | Local-model equivalent |
|---|---|---|
| Adaptive chunking | Custom regex for code fences, tables, headers | `langchain_text_splitters.MarkdownHeaderTextSplitter` + `RecursiveCharacterTextSplitter` |
| Embedding selection | `fake_embed()` hash-pooling function | `langchain_huggingface.HuggingFaceEmbeddings` (`sentence-transformers/all-MiniLM-L6-v2` or `bge-small-en-v1.5`) |
| Vector indexing | Real FAISS (already used as-is in the course) | Same — FAISS is already the course's real tool here |

### 1. Adaptive Chunking with Real Text Splitters

The course's hand-rolled `adaptive_chunk_v2` function (heading-path tracking + code/table
detection) demonstrates the *concept* precisely so you can read every line of the logic.
In practice, LangChain's splitters implement the structure-aware half of this well:

In [6]:
document_text = """# Setup

Some intro text about setup.

## Authentication

Explains how authentication works in general.

### Token Refresh

Here is how you refresh a token.

```python
def refresh(token):
    return new_token(token)
```"""

In [7]:
# OPTIONAL / ILLUSTRATIVE -- requires `langchain-text-splitters`; not executed here.
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

headers_to_split_on = [("#", "h1"), ("##", "h2"), ("###", "h3")]
header_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
header_sections = header_splitter.split_text(document_text)  # each section carries its heading path in .metadata

child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
final_chunks = []
for section in header_sections:
    for chunk in child_splitter.split_text(section.page_content):
        final_chunks.append({"text": chunk, "heading_path": section.metadata})


In [8]:
final_chunks

[{'text': 'Some intro text about setup.', 'heading_path': {'h1': 'Setup'}},
 {'text': 'Explains how authentication works in general.',
  'heading_path': {'h1': 'Setup', 'h2': 'Authentication'}},
 {'text': 'Here is how you refresh a token.  \n```python\ndef refresh(token):\nreturn new_token(token)\n```',
  'heading_path': {'h1': 'Setup',
   'h2': 'Authentication',
   'h3': 'Token Refresh'}}]

**Note:** neither of these splitters natively detects code fences or tables as atomic
units the way the course's exercise asks you to build — you'd still want the course's
fence/table-boundary detection layered on top, since this is a genuine gap in the
standard library splitters, not something the course invented unnecessarily.

### 2. Local Embeddings

In [9]:
# OPTIONAL / ILLUSTRATIVE -- requires `langchain-huggingface` and a one-time model download; not executed here.
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",   # ~130MB, downloads once, then fully offline
    encode_kwargs={"normalize_embeddings": True},
)

vectors = embeddings.embed_documents([c["text"] for c in final_chunks])


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

### 3. Indexing — Already Real in the Course

Lesson 2's FAISS index-selection exercises (HNSW vs. IVF-PQ, the decision-tree advisor)
already use real `faiss-cpu` end to end — nothing changes here when you swap in real
embeddings. The only difference is the input vectors now come from a real
384-dimensional sentence-transformer rather than a 128-dimensional hash function, so
re-tune `M`, `efConstruction`, and `nlist` for the new dimensionality and dataset size
rather than reusing the course's exact parameter values.

### What This Buys You, and What It Costs

**Pros:**
- No API key, no per-embedding cost, no rate limits — you can re-embed a 10,000-document corpus as many times as you want while iterating on chunking strategy
- Genuinely better retrieval quality than the course's hash-based stand-in, since real embeddings capture actual semantic similarity
- Deterministic given a fixed model version, keeping the spirit of the course's reproducible verification

**Cons:**
- A real, one-time model download (though small for `all-MiniLM-L6-v2` or `bge-small-en-v1.5` — under 150MB)
- Slower to embed a large corpus on CPU than the course's near-instant hash function — budget real wall-clock time for the "embed 10,000 documents" take-home extension mentioned in the README
- Still meaningfully lower quality than the largest hosted embedding models, particularly on domain-specific or multilingual text (Lesson 2's own embedding-selection lesson applies here directly)

### Bridging Back to the Course

Everything Lesson 2 teaches about *content-type-aware representation* (books vs. papers
vs. tweets vs. metadata tags) and *index selection by scale/memory* transfers unchanged
— a real embedding model still needs the same preprocessing decisions the course's
`prepare_for_embedding` function makes explicit. What changes is embedding quality and
compute time, not the architectural reasoning.

---

## Extension B — Cloud/API Models Approach

**Advanced Chunking, Embeddings & Indexing**

This section shows the real *cloud/API* tooling equivalents for embeddings and indexing.

### What Changes, Component by Component

| Course concept | Course's stand-in | Cloud/API equivalent |
|---|---|---|
| Adaptive chunking | Custom regex for code fences, tables, headers | Same splitters as the local approach — chunking itself has no cloud dependency |
| Embedding selection | `fake_embed()` hash-pooling function | `langchain_openai.OpenAIEmbeddings` (`text-embedding-3-small` / `-large`) |
| Vector indexing | Real FAISS (already used as-is) | FAISS locally, **or** a managed cloud vector store (Pinecone, Weaviate Cloud, Milvus Cloud) |

### 1. Chunking Is Cloud-Agnostic

Chunking is a pure text-processing step — there is no meaningful "cloud version" of it.
Use the same `MarkdownHeaderTextSplitter` / `RecursiveCharacterTextSplitter` combination
described in the Local Models extension above regardless of which embedding/generation
path you choose.

### 2. Cloud Embeddings

In [10]:
# OPTIONAL / ILLUSTRATIVE -- requires `langchain-openai` and a funded OPENAI_API_KEY; not executed here.
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectors = embeddings.embed_documents([c["text"] for c in final_chunks])


This is the same call pattern the Lesson 1 cloud extension uses for its retriever —
one embeddings client can serve every module.

### 3. Indexing — Local FAISS vs. a Managed Cloud Vector Store

FAISS remains a perfectly valid choice even with cloud-sourced embeddings — you can
compute vectors via `OpenAIEmbeddings` and still store/search them in a local
`faiss-cpu` index, exactly as the Lesson 1 cloud extension does. The genuinely "cloud"
alternative is a **managed vector database**, which trades local index-tuning control
(Lesson 2's whole HNSW/IVF-PQ decision tree) for a hosted service that handles scaling,
replication, and index maintenance for you:

In [17]:
from pinecone.grpc import PineconeGRPC as Pinecone
from pinecone import ServerlessSpec
pc = Pinecone()

index_name = "first-pinecone-example"

if not pc.has_index(index_name):
  pc.create_index(
    name=index_name,
    dimension=1536,
    metric="cosine",
    spec=ServerlessSpec(
      cloud="aws",
      region="us-east-1"
    )
  )

In [18]:
# OPTIONAL / ILLUSTRATIVE -- requires `langchain-pinecone`, a Pinecone account/API key, and `embeddings` from above; not executed here.
from langchain_pinecone import PineconeVectorStore

vector_store = PineconeVectorStore.from_texts(
    [c["text"] for c in final_chunks],
    embedding=embeddings,
    index_name=index_name,
)


**Trade-off:** a managed store removes the need to ever run Lesson 2's FAISS
index-selection decision tree yourself — the provider handles it — but it also means
the specific mechanics that module teaches (HNSW `M`/`efConstruction`, IVF `nlist`/PQ
compression) become someone else's implementation detail rather than a lever you
control directly. Understanding them is still valuable for reasoning about a managed
store's cost and latency characteristics, even if you never call `faiss.IndexIVFPQ`
yourself again.

### What This Buys You, and What It Costs

**Pros:**
- No local model download, no GPU/CPU embedding compute time — offload it entirely to the provider
- `text-embedding-3-large` in particular offers meaningfully higher retrieval quality than small local models on difficult, domain-specific, or multilingual corpora — directly relevant to Lesson 2's embedding-selection lesson
- A managed vector store additionally removes index operations (scaling, backups, replica management) from your team's plate

**Cons:**
- Real, ongoing per-token embedding cost — re-embedding a 10,000-document corpus (the README's take-home extension) is a one-time cost worth estimating before you run it, using the exact cost-math example from Lesson 2's embedding-selection lesson
- Every embedding call requires network access and is subject to provider rate limits — batch-embedding a large corpus needs a retry/backoff strategy that the course's instantaneous local hash function never had to model
- A managed vector store adds a genuine new failure mode (a third-party outage) and typically a recurring subscription cost, on top of the embedding API cost itself

### Bridging Back to the Course

Lesson 2's core argument — that embedding *representation* (how you serialize a table,
a tweet, or a metadata tag before embedding) matters as much as which model you choose —
holds regardless of whether the model is local or hosted. The cloud path mainly changes
the cost/quality trade-off curve and adds network/rate-limit considerations that the
course's offline labs don't need to model, but which any real production deployment
must.